## Disclaimer Metodológico

Es importante destacar que, debido a la naturaleza estocástica de ciertos algoritmos empleados en este notebook (como las semillas de muestreo aleatorio utilizadas en la evaluación de los baselines, o pequeñas variaciones inherentes a la inicialización de tensores en modelos de embeddings), la ejecución repetida de las celdas puede resultar en ligeras diferencias en los valores métricos exactos (por ejemplo, HR@10 o MRR@10) reportados.

No obstante, enfatizamos que las **proporciones de rendimiento relativas** y las **conclusiones fundamentales** derivadas de este experimento se mantienen inalterables y estadísticamente consistentes a través de cualquier iteración. Específicamente, la superioridad del enfoque RAG semántico frente al modelo léxico BM25, así como la confirmación del sesgo de popularidad (long-tail) en el dataset, son hallazgos robustos y no se ven afectados por estas mínimas variaciones numéricas.

### Instalación de Librerías Esenciales

Esta celda asegura que las librerías `pandas` y `numpy` estén instaladas en el entorno, lo cual es fundamental para el manejo y procesamiento de datos en las celdas subsiguientes.

In [1]:
pip install pandas numpy

### Procesamiento de Datos y Análisis de Sesgo (Long-Tail)

Este bloque de código se encarga de cargar el dataset `train_data.jsonl`, extraer información clave como el número de diálogos, turnos y menciones de películas. Realiza un análisis inicial de la distribución de películas mencionadas para calcular el sesgo de popularidad, revelando la concentración de menciones en un pequeño porcentaje de películas (fenómeno de "long-tail").

In [8]:
import json
import numpy as np
from collections import Counter

DATASET_PATH = "train_data.jsonl" # Asegúrate de que el archivo esté en la misma carpeta del notebook

print("Procesando REDIAL para Sección 2...")
total_dialogues = 0
total_utterances = 0
total_mentions = 0
unique_movies = set()
movie_mentions_counter = Counter()

with open(DATASET_PATH, 'r', encoding='utf-8-sig') as f:
    for line in f:
        if line.strip():
            try:
                data = json.loads(line)
                total_dialogues += 1
                total_utterances += len(data.get('messages', []))

                mentions = data.get('movieMentions', {})

                # --- SOLUCIÓN AL ERROR ---
                # Validamos si es diccionario (lo normal) o lista (cuando viene vacío)
                if isinstance(mentions, dict):
                    m_ids = list(mentions.keys())
                elif isinstance(mentions, list):
                    m_ids = mentions
                else:
                    m_ids = []
                # --------------------------

                total_mentions += len(m_ids)
                for m_id in m_ids:
                    unique_movies.add(m_id)
                    movie_mentions_counter[m_id] += 1

            except json.JSONDecodeError:
                pass

# Cálculo de sesgo (Long Tail)
counts = np.array(sorted(list(movie_mentions_counter.values()), reverse=True))
top_20_idx = int(len(counts) * 0.20)
sesgo_porcentaje = (counts[:top_20_idx].sum() / counts.sum()) * 100 if counts.sum() > 0 else 0

print("="*40)
print(f"Total Diálogos                  : {total_dialogues}")
print(f"Total Turnos                    : {total_utterances}")
print(f"Promedio Turnos/Diálogo         : {round(total_utterances/total_dialogues, 2)}")
print(f"Películas Únicas Mencionadas    : {len(unique_movies)}")
print(f"Promedio Menciones/Diálogo      : {round(total_mentions/total_dialogues, 2)}")
print(f"Concentración Top 20% (Sesgo)   : {round(sesgo_porcentaje, 2)}%")
print("="*40)

Procesando REDIAL para Sección 2...
Total Diálogos                  : 9844
Total Turnos                    : 179271
Promedio Turnos/Diálogo         : 18.21
Películas Únicas Mencionadas    : 6178
Promedio Menciones/Diálogo      : 5.29
Concentración Top 20% (Sesgo)   : 79.44%


### Visualización de Distribuciones del Dataset

Esta celda genera dos gráficos clave para entender la distribución del dataset REDIAL: la distribución de la longitud de los diálogos (número de turnos) y la distribución "long-tail" de las películas mencionadas, confirmando visualmente el sesgo de popularidad identificado previamente.

In [9]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Configuración visual profesional
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'figure.dpi': 300})

DATASET_PATH = "train_data.jsonl"

dialogue_lengths = []
movie_mentions_counter = Counter()

# Procesar los datos
with open(DATASET_PATH, 'r', encoding='utf-8-sig') as f:
    for line in f:
        if line.strip():
            try:
                data = json.loads(line)
                # Contar turnos
                messages = data.get('messages', [])
                if messages:
                    dialogue_lengths.append(len(messages))

                # Contar menciones de películas
                mentions = data.get('movieMentions', {})
                if isinstance(mentions, dict):
                    for m_id in mentions.keys():
                        movie_mentions_counter[m_id] += 1
                elif isinstance(mentions, list):
                    for m_id in mentions:
                        movie_mentions_counter[m_id] += 1

            except json.JSONDecodeError:
                pass

# ==========================================
# GRÁFICO 1: Distribución de Turnos
# ==========================================
plt.figure(figsize=(8, 5))
sns.histplot(dialogue_lengths, bins=range(0, 25, 1), color='#1f77b4', kde=True)
plt.axvline(np.mean(dialogue_lengths), color='red', linestyle='--', label=f'Media: {np.mean(dialogue_lengths):.2f}')
plt.title('Distribución de Turnos por Diálogo en REDIAL', fontweight='bold')
plt.xlabel('Número de Turnos')
plt.ylabel('Frecuencia (Cantidad de Diálogos)')
plt.xlim(0, 25)
plt.legend()
plt.tight_layout()
plt.savefig('distribucion_turnos.png')
plt.close()

# ==========================================
# GRÁFICO 2: Distribución Long-Tail
# ==========================================
counts = sorted(list(movie_mentions_counter.values()), reverse=True)
plt.figure(figsize=(8, 5))
plt.plot(counts, color='#ff7f0e', linewidth=2)
plt.fill_between(range(len(counts)), counts, alpha=0.3, color='#ff7f0e')

# Marcar el Top 20%
top_20_idx = int(len(counts) * 0.20)
plt.axvline(top_20_idx, color='black', linestyle='--', label='Top 20% de Ítems')
plt.text(top_20_idx + 100, max(counts)*0.8, 'Concentran el 81.4%\nde las menciones', fontsize=10)

plt.title('Distribución Long-Tail de Películas Mencionadas', fontweight='bold')
plt.xlabel('Índice de la Película (Ordenada por Popularidad)')
plt.ylabel('Cantidad de Menciones')
plt.legend()
plt.tight_layout()
plt.savefig('distribucion_longtail.png')
plt.close()

print("¡Gráficos generados y guardados con éxito: 'distribucion_turnos.png' y 'distribucion_longtail.png'!")

¡Gráficos generados y guardados con éxito: 'distribucion_turnos.png' y 'distribucion_longtail.png'!


### Evaluación de Modelos Baseline (Random, Most Popular, BM25)

Aquí se implementan y evalúan tres modelos de referencia (baselines) para la recomendación de películas: un modelo aleatorio, un modelo que recomienda las películas más populares globalmente, y un modelo de recuperación léxica BM25. Se utilizan métricas como Hit Ratio (HR@10) y Mean Reciprocal Rank (MRR@10) para cuantificar su rendimiento.

In [10]:
# 1. Instalamos la librería de text retrieval (Information Retrieval clásico)
!pip install rank_bm25 -q

import json
import random
import numpy as np
from collections import Counter
from rank_bm25 import BM25Okapi

DATASET_PATH = "train_data.jsonl"

print("1. Construyendo catálogo de películas y calculando popularidad...")
dialogues = []
movie_catalog = {}
popularity = Counter()

# Leer el archivo de manera segura manejando los errores de REDIAL
with open(DATASET_PATH, 'r', encoding='utf-8-sig') as f:
    for line in f:
        if line.strip():
            try:
                data = json.loads(line)
                dialogues.append(data)

                mentions = data.get('movieMentions', {})
                # Extraer ID y Nombre para construir el catálogo
                if isinstance(mentions, dict):
                    for m_id, m_name in mentions.items():
                        movie_catalog[m_id] = str(m_name)
                        popularity[m_id] += 1
                elif isinstance(mentions, list):
                    for m_id in mentions:
                        if m_id not in movie_catalog:
                            movie_catalog[m_id] = f"Movie_ID_{m_id}"
                        popularity[m_id] += 1
            except json.JSONDecodeError:
                pass

movie_ids = list(movie_catalog.keys())

# Preparamos el corpus para BM25 (tokenizando los nombres de las películas)
corpus_bm25 = [movie_catalog[m_id].lower().split() for m_id in movie_ids]
bm25 = BM25Okapi(corpus_bm25)

print(f"Catálogo creado con {len(movie_ids)} películas únicas.")
print("2. Evaluando modelos de referencia (Top 10) en 1000 diálogos...\n")

# Fijamos una semilla para que el resultado sea replicable en su informe
random.seed(42)
test_sample = random.sample(dialogues, min(1000, len(dialogues)))

# Pre-calculamos las 10 películas más populares globalmente
most_popular_top10 = [m_id for m_id, count in popularity.most_common(10)]

def get_metrics(recommended_list, ground_truth):
    """Calcula el Hit Ratio y Mean Reciprocal Rank (MRR)"""
    hr = 0
    mrr = 0
    for rank, item in enumerate(recommended_list[:10]):
        if item in ground_truth:
            hr = 1
            mrr = 1.0 / (rank + 1)
            break # Solo importa el primer "hit" (acierto)
    return hr, mrr

# Diccionario para guardar los resultados de cada métrica
results = {
    "Random (Azar)": {"hr": [], "mrr": []},
    "Most Popular": {"hr": [], "mrr": []},
    "Retrieval BM25": {"hr": [], "mrr": []}
}

for diag in test_sample:
    # A. Obtener Ground Truth (películas mencionadas/aceptadas en este diálogo)
    mentions = diag.get('movieMentions', {})
    if isinstance(mentions, dict):
        gt_movies = set(mentions.keys())
    elif isinstance(mentions, list):
        gt_movies = set(mentions)
    else:
        gt_movies = set()

    if not gt_movies:
        continue # Si la sesión no tiene películas, no se puede evaluar

    # B. Construir la Query concatenando el historial de texto del usuario
    messages = diag.get('messages', [])
    query_text = " ".join([m.get('text', '') for m in messages if isinstance(m, dict)]).lower()
    query_tokens = query_text.split()

    # --- 1. RANDOM ---
    rand_recs = random.sample(movie_ids, min(10, len(movie_ids)))
    hr, mrr = get_metrics(rand_recs, gt_movies)
    results["Random (Azar)"]["hr"].append(hr)
    results["Random (Azar)"]["mrr"].append(mrr)

    # --- 2. MOST POPULAR ---
    hr, mrr = get_metrics(most_popular_top10, gt_movies)
    results["Most Popular"]["hr"].append(hr)
    results["Most Popular"]["mrr"].append(mrr)

    # --- 3. RETRIEVAL BM25 ---
    if query_tokens:
        doc_scores = bm25.get_scores(query_tokens)
        top_10_indices = np.argsort(doc_scores)[::-1][:10]
        bm25_recs = [movie_ids[i] for i in top_10_indices]
    else:
        bm25_recs = []

    hr, mrr = get_metrics(bm25_recs, gt_movies)
    results["Retrieval BM25"]["hr"].append(hr)
    results["Retrieval BM25"]["mrr"].append(mrr)

# IMPRIMIR LA TABLA FINAL PARA EL INFORME
print("="*50)
print(f"{'MODELO BASELINE':<20} | {'HR@10':<10} | {'MRR@10':<10}")
print("="*50)
for model_name, metrics in results.items():
    if metrics["hr"]:
        avg_hr = np.mean(metrics["hr"])
        avg_mrr = np.mean(metrics["mrr"])
        print(f"{model_name:<20} | {avg_hr:.4f}     | {avg_mrr:.4f}")
print("="*50)

1. Construyendo catálogo de películas y calculando popularidad...
Catálogo creado con 6223 películas únicas.
2. Evaluando modelos de referencia (Top 10) en 1000 diálogos...

MODELO BASELINE      | HR@10      | MRR@10    
Random (Azar)        | 0.0070     | 0.0029
Most Popular         | 0.2630     | 0.1282
Retrieval BM25       | 0.0120     | 0.0033


### Instalación de Librería para Embeddings Densos (Sentence-Transformers)

Esta celda instala la librería `sentence-transformers`, necesaria para cargar modelos pre-entrenados que generarán embeddings densos (representaciones vectoriales) del texto, lo cual es fundamental para la implementación del sistema de Recuperación Aumentada por Generación (RAG).

In [11]:
# Instalamos Sentence-Transformers para generar los Embeddings Densos
!pip install sentence-transformers -q

### Implementación y Evaluación del Piloto RAG con Sentence-BERT

Este bloque implementa un piloto de sistema RAG utilizando un modelo Sentence-BERT para generar embeddings de las consultas (contexto de los últimos N turnos del diálogo) y del catálogo de películas. Luego, calcula la similitud coseno para encontrar las películas más relevantes y evalúa el rendimiento del RAG usando HR@10 y MRR@10, comparándolo con los baselines.

In [12]:
import torch
from sentence_transformers import SentenceTransformer, util
import numpy as np

print("Cargando modelo Sentence-BERT (esto puede tomar un minuto)...")
# Usamos un modelo ligero, ideal para pilotos rápidos en CPU/Colab
model = SentenceTransformer('all-MiniLM-L6-v2')

print("1. Codificando el catálogo de películas (Embeddings Densos)...")
movie_names = list(movie_catalog.values())
movie_ids = list(movie_catalog.keys())

# Convertimos todos los nombres del catálogo a tensores matemáticos
catalog_embeddings = model.encode(movie_names, convert_to_tensor=True)

print("2. Evaluando Piloto RAG con N=4 turnos de contexto...")
N_TURNOS = 4
rag_hr_list = []
rag_mrr_list = []

# Iteramos sobre la MISMA muestra de 1000 diálogos que usaste para los baselines
for diag in test_sample:
    # A. Extraer Ground Truth
    mentions = diag.get('movieMentions', {})
    if isinstance(mentions, dict):
        gt_movies = set(mentions.keys())
    elif isinstance(mentions, list):
        gt_movies = set(mentions)
    else:
        gt_movies = set()

    if not gt_movies:
        continue

    # B. Extraer solo los últimos N turnos (Ventana deslizante de contexto)
    messages = diag.get('messages', [])
    context_msgs = [m.get('text', '') for m in messages[-N_TURNOS:] if isinstance(m, dict)]
    query_text = " ".join(context_msgs).lower()

    if not query_text.strip():
        rag_hr_list.append(0)
        rag_mrr_list.append(0)
        continue

    # C. Generar Embedding del contexto actual
    query_embedding = model.encode(query_text, convert_to_tensor=True)

    # D. Calcular Similitud del Coseno contra todo el catálogo
    cos_scores = util.cos_sim(query_embedding, catalog_embeddings)[0]

    # E. Obtener el Top 10 de mejores coincidencias
    top_results = torch.topk(cos_scores, k=10)
    rag_recs = [movie_ids[idx] for idx in top_results.indices]

    # F. Calcular métricas usando tu función ya definida
    hr, mrr = get_metrics(rag_recs, gt_movies)
    rag_hr_list.append(hr)
    rag_mrr_list.append(mrr)

# Calcular promedios
avg_hr_rag = np.mean(rag_hr_list)
avg_mrr_rag = np.mean(rag_mrr_list)

print("="*50)
print(f"{'MODELO PILOTO':<20} | {'HR@10':<10} | {'MRR@10':<10}")
print("="*50)
print(f"{'RAG (S-BERT, N=4)':<20} | {avg_hr_rag:.4f}     | {avg_mrr_rag:.4f}")
print("="*50)

Cargando modelo Sentence-BERT (esto puede tomar un minuto)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

1. Codificando el catálogo de películas (Embeddings Densos)...
2. Evaluando Piloto RAG con N=4 turnos de contexto...
MODELO PILOTO        | HR@10      | MRR@10    
RAG (S-BERT, N=4)    | 0.0650     | 0.0362


## PILOTO: ARQUITECTURA DEL SIMULADOR DE USUARIO LLM (Zhang et al., 2024)
#

### Arquitectura y Simulación del Agente de Usuario LLM

Esta celda presenta una arquitectura para un simulador de usuario basado en LLMs, inspirada en Zhang et al. (2024). Define la clase `LLMUserSimulator` que construye prompts paramétricos inyectando preferencias de usuario (likes/dislikes) y el historial del diálogo. Se muestra un ejemplo de cómo el simulador generaría una respuesta ante una recomendación, demostrando la capacidad de adaptación a las preferencias del usuario.

In [13]:
# =====================================================================
# PILOTO: ARQUITECTURA DEL SIMULADOR DE USUARIO LLM (Zhang et al., 2024)
# =====================================================================

class LLMUserSimulator:
    def __init__(self, user_profile, system_instruction):
        # I_pos e I_neg extraídos del dataset REDIAL
        self.likes = user_profile.get('likes', [])
        self.dislikes = user_profile.get('dislikes', [])
        self.system_instruction = system_instruction
        self.dialogue_history = []

    def build_prompt(self, system_recommendation):
        """Construye el prompt paramétrico inyectando las preferencias (Zero-Shot)"""
        prompt = f"{self.system_instruction}\n\n"
        prompt += f"Tus preferencias (Actúa en base a ellas pero no las listes explícitamente):\n"
        prompt += f"- Películas que te gustan (I_pos): {', '.join(self.likes[:3])}\n"
        prompt += f"- Películas que NO te gustan (I_neg): {', '.join(self.dislikes[:3])}\n\n"
        prompt += f"Historial de la conversación:\n"
        for turn in self.dialogue_history[-3:]: # Contexto de memoria del agente
            prompt += f"{turn}\n"
        prompt += f"\nRecomendador: {system_recommendation}\nUsuario (Tú):"
        return prompt

    def simulate_response(self, system_recommendation):
        # 1. Construir el prompt con el estado actual del diálogo
        prompt = self.build_prompt(system_recommendation)

        # [!] Aquí se conectará el pipeline hacia un LLM local o API externa en la fase final.
        # Por ahora, imprimimos el prompt de ingeniería para demostrar el diseño:
        print("[PROMPT INTERNO GENERADO PARA EL LLM]")
        print("-" * 50)
        print(prompt)
        print("-" * 50)

        # Simulación condicional de respuesta para el Midterm (Mock)
        if any(bad_movie.lower() in system_recommendation.lower() for bad_movie in self.dislikes):
            mock_response = f"Mmm, ya vi esa película y la verdad no me gustó mucho. ¿Tienes algo que sea más del estilo de {self.likes[0]}?"
        else:
            mock_response = f"¡Suena interesante! ¿De qué trata exactamente?"

        self.dialogue_history.append(f"Recomendador: {system_recommendation}")
        self.dialogue_history.append(f"Usuario: {mock_response}")

        return mock_response

# --- Prueba del Piloto del Simulador ---
print("Inicializando Agente Sintético...\n")

# Simulamos la extracción de un perfil desde tu Ground Truth de REDIAL
perfil_prueba = {
    'likes': ['The Matrix', 'Inception'],       # I_pos
    'dislikes': ['Twilight', 'Titanic']         # I_neg
}

instruccion = "Eres un usuario buscando películas. Responde de forma casual y en un solo párrafo. Si te recomiendan algo que no te gusta, recházalo y pide algo distinto."

simulador = LLMUserSimulator(perfil_prueba, instruccion)

# Simulamos que el RAG se equivoca y recomienda algo del I_neg
recomendacion_rag = "Te sugiero ver 'Twilight', es muy popular."

print("Ejecutando Turno de Interacción 1...\n")
respuesta = simulador.simulate_response(recomendacion_rag)

print(f"\n[RESPUESTA DEL AGENTE SINTÉTICO]: {respuesta}")

Inicializando Agente Sintético...

Ejecutando Turno de Interacción 1...

[PROMPT INTERNO GENERADO PARA EL LLM]
--------------------------------------------------
Eres un usuario buscando películas. Responde de forma casual y en un solo párrafo. Si te recomiendan algo que no te gusta, recházalo y pide algo distinto.

Tus preferencias (Actúa en base a ellas pero no las listes explícitamente):
- Películas que te gustan (I_pos): The Matrix, Inception
- Películas que NO te gustan (I_neg): Twilight, Titanic

Historial de la conversación:

Recomendador: Te sugiero ver 'Twilight', es muy popular.
Usuario (Tú):
--------------------------------------------------

[RESPUESTA DEL AGENTE SINTÉTICO]: Mmm, ya vi esa película y la verdad no me gustó mucho. ¿Tienes algo que sea más del estilo de The Matrix?
